*Módulo 9 de 9*

> **Prefer English?** Open [`09_from_browser_to_production.ipynb`](../en/09_from_browser_to_production.ipynb) — it is the same module, in English.


# 🚀 Módulo 9 — De tu navegador a producción

🧭 **Objetivos** — conectar todo lo que hiciste en el navegador con la
herramienta de producción real,
**[geocrop_analysis_mx](https://github.com/abxda/geocrop_analysis_mx)**.
Entender qué cambia a escala, qué *no puede* hacer el navegador y por qué, y
exactamente cómo correr tú mismo el pipeline de producción con los mismos
datos del Valle del Yaqui — offline, sin necesidad de Google Earth Engine.

Ahora estás equipado para usar esa herramienta de forma *consciente*: sabes
qué hace cada fase, qué necesita, y cómo juzgar su resultado.

![pipeline completo](../../anim/es/10_full_pipeline.svg)


## La misma historia, todo más grande

Cada paso que aprendiste corresponde uno a uno con una fase del pipeline de
producción — solo que corre más grande:

| Tú hiciste (navegador) | `geocrop_analysis_mx` (escritorio) | Fase |
|---|---|---|
| 1 tile, 1 mes | regiones completas, **muchos meses** | `download` |
| geomediana ya dada | construye geomedianas desde STAC/COG abierto **o** GEE | `download` |
| solo bandas ópticas | **+ radar Sentinel-1 (SAR)** y RVI | `download` |
| `shepherd-wasm` (NumPy) | `pyshepseg` (acelerado con numba) | `segment` |
| filtro de pureza, a mano | unión espacial + filtro de pureza | `label` |
| media/desv de 13 capas | estadística zonal completa sobre **~846 variables** | `extract` |
| Random Forest | **TPOT (AutoML)** busca el mejor pipeline | `train` |
| pintar el arreglo | escribe un **GeoPackage** para QGIS | `predict` |

📚 **¿Por qué radar?** Las nubes bloquean los sensores ópticos; el **Radar de
Apertura Sintética** (Sentinel-1) ve a través de ellas y percibe estructura y
humedad — pistas extra, sobre todo en temporadas nubladas. 📚 **¿Por qué
muchos meses?** Esa es la **fenología** (Módulo 4) — la historia del NDVI a
lo largo de la temporada que separa cultivos parecidos. Los meses × bandas ×
índices apilados forman un **cubo de datos**.


## Dos formas de descargar las imágenes (gratis por defecto, GEE opcional)

En todo este curso nunca diste una cuenta de Google Earth Engine — y el
pipeline de producción tampoco la necesita. `geocrop_analysis_mx` tiene
**dos backends de descarga**, elegidos en el archivo de configuración:

- **`download_backend: "stac"` (por defecto, gratis, sin cuenta).** Las
  imágenes se traen directo de catálogos abiertos en la nube **STAC/COG** y
  la geomediana se calcula localmente. El proveedor óptico lo fija
  `hls_provider`:
  - `"mpc"` — **Microsoft Planetary Computer**, anónimo, sin token (el
    respaldo por defecto; su archivo HLS tiene huecos antes de ~2020).
  - `"nasa"` — **NASA LPCLOUD**, el archivo HLS completo y autoritativo;
    necesita un token *gratuito* de NASA Earthdata en `EARTHDATA_TOKEN`
    (Perfil → Generate Token en urs.earthdata.nasa.gov).
  - `"earthsearch"` — **Element 84 / AWS** Sentinel-2, anónimo y el único que
    funciona dentro de un navegador (habilitado para CORS).
  - `"auto"` — earthsearch en WASM, si no NASA si hay token, si no MPC.
  El radar Sentinel-1 siempre viene de Planetary Computer (anónimo).

- **`download_backend: "gee"` (opcional).** Si *sí* tienes cuenta de Google
  Earth Engine, la composición corre en los servidores de Google y solo
  descargas el resultado — menos CPU local, pero requiere
  `pip install earthengine-api` + `earthengine authenticate`. GEE es
  estrictamente opcional; nada de él se importa a menos que lo actives.

Así que la vía gratis (NASA / Planetary Computer / AWS) es la primaria; GEE
es una comodidad para quien ya la tiene. Hay una guía paso a paso de los
tokens gratuitos en `docs/manual_tokens_gratuitos.pdf` del repo.


## Qué no puede hacer el navegador — y por qué

Ser honesto sobre los límites es parte de usar bien la herramienta. El
navegador (WebAssembly / Pyodide) es maravilloso para *aprender* y trabajos
*pequeños*, pero:

- **Memoria.** El Python del navegador es **wasm32**: un techo duro de unos
  **4 GB**. Un estado completo o un cubo de muchos meses no cabe; necesitarías
  teselado y streaming. La herramienta de escritorio tiene toda la RAM de tu
  máquina.
- **Sin numba, sin binarios GDAL.** Las librerías rápidas de escritorio
  (`pyshepseg` con numba, `earthengine-api`, la búsqueda paralela de TPOT) o
  no existen en Pyodide o corren mucho más lento. Por *eso* existe
  `shepherd-wasm` — un port en NumPy puro para que al menos la segmentación
  corra en el navegador.
- **Tiempo de cómputo.** El AutoML sobre cientos de variables en una región
  son de minutos a horas de CPU — bien en escritorio, doloroso en una pestaña.

**La regla práctica:** aprende y prototipa en el navegador; corre regiones
reales en escritorio (o el entorno portable, a continuación). Nada de lo que
aprendiste se desperdicia — es el *mismo pipeline*, solo un motor más grande.


## Corre tú mismo el pipeline de producción (offline, sin GEE)

`geocrop_analysis_mx` se instala con `pip` a secas — **sin conda, sin cuenta
de Google Earth Engine** — y viene con los datos de prueba del Valle del
Yaqui ya pre-procesados, así que puedes reproducir todo offline. En una
terminal (no en este navegador):

```bash
git clone https://github.com/abxda/geocrop_analysis_mx
cd geocrop_analysis_mx
python -m venv .venv && . .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install -r requirements.txt

# Copia los datos de prueba del Yaqui en su lugar (activa el modo OFFLINE)
python src/main.py --config config.test.yaml --phase setup_test

# Corre el pipeline completo — detecta los mosaicos offline y salta cualquier
# descarga / conexión a GEE automáticamente:
python src/main.py --config config.test.yaml --phase full_run
```

Verás las mismas siete fases que ahora entiendes — segment, label, extract,
train (TPOT), predict — terminando con un `predicted_map_test.gpkg` y un
`classification_report.txt` que reporta alrededor de **89% de exactitud**.
Abre el GeoPackage en **QGIS** para explorar tu mapa de cultivos como datos
**vectoriales** reales, con coordenadas.

**Opcional — con GEE.** Si *sí* tienes cuenta de Google Earth Engine y
quieres clasificar tu propia área para fechas nuevas, `config.yaml` muestra
cómo apuntar a tu propia AOI y dejar que la fase `download` construya
geomedianas en vivo. GEE es una opción, nunca un requisito.


## El entorno portable (sin navegador, sin dolores de instalación)

Hay una tercera vía, entre el navegador y una instalación completa de
desarrollador: un **entorno portable** — un Python autocontenido que corre
desde una carpeta, sin permisos de administrador, sin conda. El proyecto
hermano **[portable-satelital](https://abxda.github.io/portable-satelital/)**
ya resolvió esto (un Python relocalizable y firmado más un lanzador de un
clic). La misma receta aplica aquí: distribuir `geocrop_analysis_mx` con un
Python portable para que una persona no experta pueda dar doble clic y correr
el pipeline de escritorio sin tocar una terminal. Navegador para aprender,
portable para trabajo real en una laptop, escritorio/servidor para regiones
grandes — el mismo curso, tres motores.


## 🎓 Lo lograste

Empezaste sin saber qué era la reflectancia de un píxel. Ahora entiendes — y
has *corrido* — cada paso desde la luz satelital cruda hasta un mapa de
cultivos validado, y sabes cuándo confiar en el navegador, cuándo pasar al
escritorio, y cómo leer un clasificador con honestidad. Ese es todo el
punto: no apretar un botón, sino usar `geocrop_analysis_mx` de forma
**consciente**, con conciencia de sus conceptos, sus requisitos y sus
límites.

🔭 Para seguir profundizando en cualquier concepto, sigue los enlaces
**Profundiza** de cada módulo — abren las tarjetas de conceptos bilingües de
**[rs-learning-audio](https://abxda.github.io/rs-learning-audio/)**, donde
cada idea tiene sus prerrequisitos, linaje y referencias.


## 🔭 Profundiza

Opcional: estas tarjetas bilingües de conceptos amplían lo que acabas
de aprender (prerrequisitos, linaje a fundamentos, referencias):

- [Radar de Apertura Sintética (Sentinel-1)](https://abxda.github.io/rs-learning-audio/?id=synthetic-aperture-radar&lang=es)
- [Índice de Vegetación de Radar (RVI)](https://abxda.github.io/rs-learning-audio/?id=radar-vegetation-index&lang=es)
- [El cubo de datos (espacio × bandas × tiempo)](https://abxda.github.io/rs-learning-audio/?id=data-cube&lang=es)
- [Big data en percepción remota](https://abxda.github.io/rs-learning-audio/?id=big-data&lang=es)
- [Datos geoespaciales y QGIS](https://abxda.github.io/rs-learning-audio/?id=geospatial-data&lang=es)
- [Sistemas de referencia de coordenadas](https://abxda.github.io/rs-learning-audio/?id=coordinate-reference-system&lang=es)
- [Datos ráster](https://abxda.github.io/rs-learning-audio/?id=raster&lang=es)
- [Datos vectoriales (GeoPackage)](https://abxda.github.io/rs-learning-audio/?id=vector&lang=es)



---

[← Anterior · Módulo 8 — Proyecto final: el mapa de cultivos](08_proyecto_mapa_de_cultivos.ipynb)
